# Embedding-space diagram — ViDeBERTa-pruned vs NeoBERT, before/after SALT

Two-panel scatter (one **joint PCA basis**, so panels are directly comparable):

- **Before**: raw pruned ViDeBERTa donor rows vs NeoBERT's original input embeddings — disjoint clouds (different mean/scale/orientation).
- **After**: the SALT-projected rows actually **injected into our init artifact** vs the same NeoBERT embeddings — overlapping clouds.

Anchor rows (verbatim NeoBERT copies) and special tokens are **excluded**, so the after-overlap shows the SALT local maps, not trivial copies. Reads weight files only (no model instantiation → no remote-code/rotary fragility). CPU-only, no GPU needed.

Output: `SALT3/figures/embedding_alignment_before_after_<init>.png` + printed centroid/norm stats.

In [ ]:
%%capture
!pip install -U huggingface_hub transformers safetensors pandas matplotlib

In [ ]:
import sys, importlib
from pathlib import Path

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3') if Path('/content/drive/MyDrive').exists() else Path.cwd() / 'SALT3'
sys.path.insert(0, '/content'); sys.path.insert(0, str(PROJECT_ROOT / 'code'))  # project code FIRST so a stale /content/*.py never shadows the synced modules

import salt3_embedding_alignment_viz as viz; importlib.reload(viz)

INIT_ROOT = PROJECT_ROOT / 'init'
available = sorted(p.name for p in INIT_ROOT.iterdir() if (p / 'model' / 'model.safetensors').exists())
print('Available init artifacts:')
for name in available:
    print(' -', name)

# Default to the winning SALT arm (per-token decoder); the encoder embeddings are
# identical across the 01c decoder-variant arms, so any SALT arm gives the same figure.
INIT_NAME = next((n for n in available if 'salt' in n and 'pertoken' in n),
                 next((n for n in available if 'salt' in n), available[0]))
print('\nUsing INIT_NAME =', INIT_NAME, ' (override this variable to pick another arm)')

In [ ]:
FIG_PATH = PROJECT_ROOT / 'figures' / f'embedding_alignment_before_after_{INIT_NAME}.png'
stats = viz.run_diagram(INIT_ROOT / INIT_NAME, save_path=FIG_PATH)

shrink = stats['centroid_dist_before'] / max(stats['centroid_dist_after'], 1e-9)
print(f"\nCentroid distance shrank {shrink:.1f}x (before {stats['centroid_dist_before']:.2f} -> after {stats['centroid_dist_after']:.2f}); "
      f"row-norm: neo {stats['row_norm_neo']:.2f}, before {stats['row_norm_before']:.2f}, after {stats['row_norm_after']:.2f}")